# Практика: Векторные БД и Retrieval Augmented Generation

## Цели практики

В лекции мы обсуждали архитектуру векторных баз данных (HNSW, IVF), стратегии чанкинга и метрики оценки. Теперь мы соберем полноценный RAG-пайплайн с нуля, чтобы увидеть, как эти концепции работают в коде.

**Легенда:**
Вы — AI Engineer в быстрорастущей компании **"РобоТех"**. В корпоративной базе знаний накопился "культурный слой": актуальные регламенты за 2026 год смешались с архивными документами 2024 года. Сотрудники жалуются, что старый чат-бот путается в показаниях и выдает устаревшую информацию.

**Ваша задача:**
Создать надежную систему поиска и генерации ответов, которая:
1.  Умеет отличать актуальные документы от архивных (**Metadata Filtering**).
2.  Использует современные эмбеддинги для поиска по смыслу (**Dense Retrieval**).
3.  Генерирует точные ответы без галлюцинаций (**RAG**).
4.  Автоматически оценивает свое качество (**LLM-as-a-Judge**).

**🛠 Технический стек:**
*   **Vector DB:** Qdrant (локальный запуск)
*   **LLM:** Qwen3 8B Q4_K_M (через Ollama)
*   **Embeddings:** BAAI/bge-m3 (SOTA multilingual)
*   **Framework:** LangChain

---

## Импорты и настройка окружения

In [ ]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any

# Инструменты RAG
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams

# Визуализация
import pandas as pd
from IPython.display import display, Markdown

# Конфиг
DATA_DIR = Path("../data/knowledge_base")
COLLECTION_NAME = "robotex_docs"
QDRANT_URL = "http://localhost:6333"
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_DIMENSION = 1024

# Проверка доступного ускорителя
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
import torch

if torch.backends.mps.is_available():
    device = "mps"  # Apple Silicon / Metal
elif torch.cuda.is_available():
    device = "cuda"  # NVIDIA GPU
else:
    device = "cpu"

print(f"🚀 Device: {device.upper()}")
if device == "mps":
    print("   Accelerator: Apple Metal Performance Shaders")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## Загрузка и "хитрый" парсинг

In [2]:
def load_and_enrich_documents(directory: str) -> List[Any]:
    """
    Загружает MD файлы и извлекает метаданные из текста (эмуляция парсинга заголовков).
    """
    loader = DirectoryLoader(directory, glob="*.md", loader_cls=TextLoader)
    raw_docs = loader.load()
    
    enriched_docs = []
    for doc in raw_docs:
        content = doc.page_content
        meta = doc.metadata
        
        # 1. Извлекаем Год (ищем "2024", "2025", "2026")
        # Если года нет явно, считаем 2026 (текущий)
        years = re.findall(r"202[4-6]", content)
        meta["year"] = int(years[0]) if years else 2026
        
        # 2. Определение категории по имени файла
        filename = Path(meta["source"]).name
        if "HR" in filename or "Holiday" in filename:
            meta["category"] = "HR"
        elif "Security" in filename or "VPN" in filename:
            meta["category"] = "IT_Security"
        elif "Project" in filename:
            meta["category"] = "Projects"
        else:
            meta["category"] = "General"
            
        enriched_docs.append(doc)
        
    print(f"✅ Загружено {len(enriched_docs)} документов.")
    return enriched_docs

# Запускаем
docs = load_and_enrich_documents(str(DATA_DIR))

# Посмотрим на пример данных
df = pd.DataFrame([d.metadata for d in docs])
display(df.head())

✅ Загружено 30 документов.


,source,year,category
0,../data/knowledge_base/Doc_Filler_20.md,2026,General
1,../data/knowledge_base/Doc_Filler_13.md,2026,General
2,../data/knowledge_base/Doc_Filler_21.md,2026,General
3,../data/knowledge_base/Doc_Filler_24.md,2026,General
4,../data/knowledge_base/Doc_Filler_5.md,2026,General


## Чанкинг (Стратегия Recursive)

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Достаточно крупно, чтобы сохранить смысл абзаца
    chunk_overlap=200,    # Нахлест, чтобы не терять контекст на границах
    separators=["\n## ", "\n\n", "\n", " ", ""] # Приоритет разделителей (заголовки -> абзацы)
)

splits = text_splitter.split_documents(docs)
print(f"📚 Исходных документов: {len(docs)}")
print(f"✂️  Получено чанков: {len(splits)}")

# Пример чанка
print("\n--- Пример чанка ---")
print(splits[0].page_content[:200] + "...")
print("Metadata:", splits[0].metadata)

📚 Исходных документов: 30
✂️  Получено чанков: 65

--- Пример чанка ---
# Онбординг нового персонала в РобоТехе

## Цель

Целью данного документа является описание процесса онбординга для новых сотрудников в компании РобоТех. Представленная инструкция предназначена для пр...
Metadata: {'source': '../data/knowledge_base/Doc_Filler_20.md', 'year': 2026, 'category': 'General'}


## Инициализация Эмбеддингов (Local Accelerator)

In [ ]:
print(f"⏳ Загрузка модели эмбеддингов {EMBEDDING_MODEL} на {device}...")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True} # Важно для косинусного расстояния!
)

# Тест векторизации
test_vec = embeddings.embed_query("Тестовый запрос")
print(f"✅ Вектор создан. Размерность: {len(test_vec)}") # Должно быть 1024 для bge-m3

## Создание коллекции в Qdrant (Low-Level Control)

In [5]:
client = QdrantClient(url=QDRANT_URL)

# Проверяем, есть ли коллекция, и пересоздаем её для чистоты эксперимента
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
    print(f"🗑️ Удалена старая коллекция {COLLECTION_NAME}")

print(f"🛠 Создание коллекции с HNSW индексом...")
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIMENSION, distance=Distance.COSINE),
    # Настройка HNSW 
    hnsw_config=models.HnswConfigDiff(
        m=16,               # Количество связей на узел (больше = точнее, но больше памяти)
        ef_construct=100    # Глубина поиска при построении индекса
    )
)
print("✅ Коллекция готова!")

🗑️ Удалена старая коллекция robotex_docs
🛠 Создание коллекции с HNSW индексом...
✅ Коллекция готова!


## Загрузка векторов (Indexing)

In [6]:
print("⏳ Индексация документов (может занять время)...")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

vector_store.add_documents(splits)

print(f"🎉 Успешно проиндексировано {len(splits)} чанков.")

⏳ Индексация документов (может занять время)...
🎉 Успешно проиндексировано 65 чанков.


## Проблема "Наивного поиска" (Naive Retrieval)

In [7]:
query = "Какие правила удаленной работы?"

print(f"🔎 Запрос: {query}\n")

# Ищем топ-3 похожих чанка без фильтров
results = vector_store.similarity_search_with_score(query, k=3)

print("--- Результаты поиска (Naive) ---")
for i, (doc, score) in enumerate(results):
    year = doc.metadata.get('year', 'N/A')
    category = doc.metadata.get('category', 'N/A')
    print(f"[{i+1}] Score: {score:.4f} | Год: {year} | Категория: {category}")
    print(f"    Текст: {doc.page_content[:150]}...\n")

🔎 Запрос: Какие правила удаленной работы?

--- Результаты поиска (Naive) ---
[1] Score: 0.7331 | Год: 2024 | Категория: HR
    Текст: Политика Удаленной Работы (Архив 2024)

### Введение

В соответствии с постоянно эvolving'ем технологий и требований к р...

[2] Score: 0.6771 | Год: 2026 | Категория: HR
    Текст: Корпорация РобоТех определяет следующие правила для гибридной работы:

1. Обязательное присутствие в офисе по Вторникам, Средам и Четвергам.
2. Работа...

[3] Score: 0.6588 | Год: 2024 | Категория: HR
    Текст: 4. **Обязанности менеджмента**
   - Менеджеры обязаны обеспечить доступность сотрудников для связи и координации работы, а также проводить мероприятия...



## Умный поиск с фильтрацией (Metadata Filtering)

In [8]:
from qdrant_client.http import models as rest

print(f"🔎 Запрос: {query} (с фильтром year=2026)\n")

# Создаем жесткий фильтр
filter_condition = rest.Filter(
    must=[
        rest.FieldCondition(
            key="metadata.year",
            match=rest.MatchValue(value=2026)
        )
    ]
)

# Выполняем поиск с фильтром
# Примечание: В LangChain для Qdrant фильтр передается именно так
filtered_results = vector_store.similarity_search_with_score(
    query, 
    k=3, 
    filter=filter_condition
)

print("--- Результаты поиска (Filtered) ---")
for i, (doc, score) in enumerate(filtered_results):
    year = doc.metadata.get('year', 'N/A')
    print(f"[{i+1}] Score: {score:.4f} | Год: {year}")
    print(f"    Текст: {doc.page_content[:150]}...\n")

# Теперь в выдаче должна быть только актуальная политика.

🔎 Запрос: Какие правила удаленной работы? (с фильтром year=2026)

--- Результаты поиска (Filtered) ---
[1] Score: 0.6771 | Год: 2026
    Текст: Корпорация РобоТех определяет следующие правила для гибридной работы:

1. Обязательное присутствие в офисе по Вторникам, Средам и Четвергам.
2. Работа...

[2] Score: 0.5772 | Год: 2026
    Текст: Политика Гибридной Работы (Действует с 2026)

Год документа: 2026
Статус: Активен

Введение
--------

В соответствии с...

[3] Score: 0.5710 | Год: 2026
    Текст: Clean Desk Policy

1. **Цель и предмет регламента**

   Целью данного регламента является установление правил организации рабочего м...



## Сборка RAG-пайплайна (Generation)

In [9]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Инициализация LLM
llm = ChatOllama(
    model="hf.co/Qwen/Qwen3-8B-GGUF:Q4_K_M", 
    temperature=0.1, # Низкая температура для фактологической точности
    base_url="http://localhost:11434"
)

# 2. Промпт (Инструкция)
# Используем шаблон, который заставляет модель опираться ТОЛЬКО на контекст.
template = """Ты — корпоративный ассистент компании "РобоТех".
Ответь на вопрос сотрудника, используя ТОЛЬКО предоставленный ниже контекст.
Если в контексте нет информации, скажи "В документах нет информации об этом".
Не придумывай факты.

Контекст:
{context}

Вопрос: {question}

Ответ:"""

prompt = ChatPromptTemplate.from_template(template)

# 3. Функция форматирования документов в строку
def format_docs(docs):
    return "\n\n".join([f"[Документ {d.metadata.get('source', 'unknown')}]\n{d.page_content}" for d in docs])

# 4. Ретривер с фильтром (зашиваем фильтр 2026 года прямо в ретривер)
# В реальной системе фильтр формируется динамически
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": filter_condition 
    }
)

# 5. Сборка цепочки (LCEL - LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("🔗 RAG Chain собрана!")

🔗 RAG Chain собрана!


## Финальный тест системы

In [10]:
question = "Сколько дней в неделю можно работать из дома и в какие дни?"

print(f"❓ Вопрос: {question}\n")
print("⏳ Генерация ответа (Qwen3 думает)...\n")

# Запуск
response = rag_chain.invoke(question)

print("🤖 Ответ ассистента:")
display(Markdown(response))

❓ Вопрос: Сколько дней в неделю можно работать из дома и в какие дни?

⏳ Генерация ответа (Qwen3 думает)...

🤖 Ответ ассистента:


 Согласно документу HR_policy_2026, сотрудники могут работать удаленно по Понедельникам и Пятницам. В офисе обязательное присутствие по Вторникам, Средам и Четвергам.

## Проверка на "галлюцинации" (Negative Test)

In [11]:
# Вопрос про то, чего нет в сгенерированных документах
fake_question = "Какая зарплата у Senior Python Developer?"

print(f"❓ Вопрос: {fake_question}")
response_fake = rag_chain.invoke(fake_question)

print("\n🤖 Ответ ассистента:")
print(response_fake)

❓ Вопрос: Какая зарплата у Senior Python Developer?

🤖 Ответ ассистента:
 В документе нет информации об этой теме.


## Настройка LLM-Судьи

In [12]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# 1. Определяем структуру ответа Судьи (через Pydantic)
class Grade(BaseModel):
    score: int = Field(description="Оценка от 1 до 5, где 5 - идеальный ответ")
    is_faithful: bool = Field(description="True, если ответ не содержит галлюцинаций и основан на контексте")
    reasoning: str = Field(description="Краткое объяснение оценки")

# 2. Парсер для преобразования ответа LLM в JSON
parser = JsonOutputParser(pydantic_object=Grade)

# 3. Промпт для Судьи
judge_template = """Ты — беспристрастный судья, оценивающий качество RAG-системы.
Проанализируй Входные данные и оцени качество Ответа Ассистента.

Входные данные:
1. ВОПРОС пользователя: {question}
2. КОНТЕКСТ (найденные документы): {context}
3. ОТВЕТ Ассистента: {answer}

Критерии оценки:
- Faithfulness: Ответ должен опираться ТОЛЬКО на предоставленный контекст. Если Ассистент добавил информацию "от себя" — это плохо.
- Relevance: Ответ должен четко отвечать на вопрос пользователя.

Формат вывода (JSON):
{format_instructions}
"""

judge_prompt = ChatPromptTemplate.from_template(
    template=judge_template,
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. Цепочка оценки
judge_chain = judge_prompt | llm | parser

print("⚖️ Судья готов к работе.")

⚖️ Судья готов к работе.


## Запуск оценки (Evaluation)

In [13]:
# Восстановим данные из прошлого успешного запуска
# question = "Сколько дней в неделю можно работать из дома и в какие дни?"
# response - это текст ответа Qwen3
# filtered_results - это документы, которые мы нашли

# Превратим список документов обратно в строку для подачи Судье
context_text = format_docs([doc for doc, score in filtered_results])

print(f"⚖️ Оценка ответа на вопрос: '{question}'...\n")

evaluation = judge_chain.invoke({
    "question": question,
    "context": context_text,
    "answer": response  # Тот ответ, который сгенерировала сеть в ячейке 10
})

print("📊 Вердикт Судьи:")
print(f"Оценка: {evaluation['score']}/5")
print(f"Достоверность (Faithful): {evaluation['is_faithful']}")
print(f"Причина: {evaluation['reasoning']}")

⚖️ Оценка ответа на вопрос: 'Сколько дней в неделю можно работать из дома и в какие дни?'...

📊 Вердикт Судьи:
Оценка: 5/5
Достоверность (Faithful): True
Причина: The response is based solely on the provided context and directly answers the user's question.


## Итоги практики

Поздравляем! Вы только что построили полноценный RAG-пайплайн, который не просто "болтает", а решает конкретную бизнес-задачу с учетом контекста и ограничений.

**Ключевые выводы:**

1.  **Векторный поиск — не серебряная пуля.**
    Как мы увидели в этапе *Naive Search*, семантическая близость не гарантирует фактологическую точность. Модель нашла документ 2024 года, потому что он был *похож* по смыслу, но он был *вреден* для ответа.

2.  **Метаданные критически важны.**
    Чистый векторный поиск (Dense Retrieval) хорош для нахождения темы, но для точного ответа в корпоративных системах необходим **Hybrid approach** (Векторы + Жесткие фильтры). Мы использовали `Payload Filtering` в Qdrant, чтобы отсечь устаревшие знания.

3.  **Архитектура имеет значение.**
    Мы не просто использовали "черный ящик", а явно настроили параметры индекса **HNSW** и выбрали специализированную модель эмбеддингов (`bge-m3`), что дало нам контроль над производительностью и качеством.

4.  **Оценка не должна быть "на глаз".**
    Внедрение паттерна **LLM-as-a-Judge** позволяет автоматизировать тестирование. Теперь, изменяя промпты или настройки чанкинга, вы можете сразу получить численную метрику качества, а не перечитывать сотни ответов вручную.

**Что можно улучшить (Home Work):**
*   Попробуйте изменить `chunk_size` в ячейке 3 и посмотрите, как изменится точность поиска.
*   Реализуйте **Hybrid Search** (Keyword + Vector), например через BM25 + dense retrieval.
*   Добавьте этап **Re-ranking** (переранжирование) результатов перед отправкой в LLM для еще большей точности.